# M2 · Entrenamiento de segmentación de instancias de vehículos (Roboflow + YOLO11-seg)

> **© 2026 Matías Darío Mercier Rengel** — Sistema Inteligente de Control de Peaje (IAM-811, UNIFRANZ). Todos los derechos reservados. Repositorio original: https://github.com/MATT-UwU/Laboratorio-final-Hito-4
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio Final de Hito: Sistema de Control de Peaje

Este notebook entrena el modelo **M2**, que segmenta cada vehículo y lo clasifica en **moto, auto, minibús, camión o bus** para determinar la tarifa.

| Paso | Dónde | Resultado |
|---|---|---|
| 1. Dataset | **Roboflow** (proyecto de tipo *Instance Segmentation*) | Versión exportable del dataset |
| 2. Entrenamiento | **Este notebook en Colab (GPU)** | `best.pt` de YOLO11-seg |
| 3. Métricas | Este notebook | mAP de cajas y máscaras, matriz de confusión |
| 4. Publicación | **Google Drive** `MyDrive/peaje_modelos/m2_best.pt` | Lo carga `servidor_inferencia_peaje_colab.ipynb` |

**Requisito de la rúbrica:** el dataset debe ser propio y estar etiquetado en Roboflow. Se puede partir de un dataset de Roboflow Universe (*Fork* al workspace), pero hay que **agregar imágenes propias** (fotos tomadas por el equipo), etiquetarlas con polígonos y generar una nueva versión.

**Antes de ejecutar:** en Colab, ve a **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**. Ten a mano el *workspace*, el *project* y la versión de Roboflow, y tu API key (Roboflow → Settings → API Keys). El notebook la pide de forma oculta y no la guarda.

## 1. Instalación y GPU

In [ ]:
!nvidia-smi -L || echo "Sin GPU: activa la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
%pip -q install -U ultralytics roboflow

In [ ]:
import os, glob, json, shutil, time, getpass
import yaml, torch, cv2, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from google.colab import drive, files

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('ADVERTENCIA: sin GPU el entrenamiento será muy lento.')

## 2. Google Drive (para conservar el modelo)
Colab borra `/content` al cerrar el runtime. El peso final y las métricas se copian a `MyDrive/peaje_modelos/`.

In [ ]:
drive.mount('/content/drive')
DIR_DRIVE = '/content/drive/MyDrive/peaje_modelos'
os.makedirs(DIR_DRIVE, exist_ok=True)
print('Carpeta de modelos:', DIR_DRIVE)

## 3. Descargar el dataset desde Roboflow
Datos del proyecto: en la URL `https://app.roboflow.com/<workspace>/<project>/<version>`.
Se exporta en formato **YOLOv11** (segmentación con polígonos).

Si la exportación YOLOv11 trae solo cajas, la celda prueba la exportación **COCO con segmentación** y, como último recurso, genera las máscaras con **SAM 2.1** a partir de las cajas. M2 siempre se entrena como segmentación.


In [ ]:
from roboflow import Roboflow
import numpy as np

RF_WORKSPACE = input('Roboflow workspace: ').strip()
RF_PROJECT   = input('Roboflow project: ').strip()
RF_VERSION   = int(input('Versión del dataset (número): ').strip())
rf_api_key   = getpass.getpass('Roboflow API key (oculta, no se guarda): ').strip()

rf = Roboflow(api_key=rf_api_key)
del rf_api_key
proyecto = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = proyecto.version(RF_VERSION)
print('Tipo de proyecto en Roboflow:', proyecto.type)

# 1) Descarga en formato YOLOv11 y revisa si las etiquetas traen polígonos.
DIR_DS = '/content/dataset_m2'
shutil.rmtree(DIR_DS, ignore_errors=True)
version.download('yolov11', location=DIR_DS)
SPLITS = [s for s in ('train', 'valid', 'test') if os.path.isdir(os.path.join(DIR_DS, s, 'images'))]
cfg_yolo = yaml.safe_load(open(os.path.join(DIR_DS, 'data.yaml'), encoding='utf-8'))
NOMBRES = cfg_yolo['names'] if isinstance(cfg_yolo['names'], list) else [cfg_yolo['names'][k] for k in sorted(cfg_yolo['names'])]

def contar_lineas(dir_ds):
    poligonos = cajas = 0
    for f in glob.glob(os.path.join(dir_ds, '*', 'labels', '*.txt')):
        for linea in open(f):
            n = len(linea.split())
            if n > 5: poligonos += 1
            elif n == 5: cajas += 1
    return poligonos, cajas

poligonos, cajas = contar_lineas(DIR_DS)
print(f'Etiquetas YOLOv11: {poligonos} polígonos, {cajas} cajas')
FUENTE_MASCARAS = None
if poligonos > cajas:
    FUENTE_MASCARAS = 'polígonos de Roboflow'

# 2) Si solo hay cajas, intenta la exportación COCO con segmentación.
def coco_a_yolo_seg(dir_coco):
    """Convierte _annotations.coco.json de cada split a etiquetas YOLO-seg dentro de DIR_DS (mismo orden de clases)."""
    convertidas = 0
    for split in SPLITS:
        ruta_json = os.path.join(dir_coco, split, '_annotations.coco.json')
        if not os.path.exists(ruta_json):
            continue
        coco = json.load(open(ruta_json, encoding='utf-8'))
        cat_a_idx = {c['id']: NOMBRES.index(c['name']) for c in coco['categories'] if c['name'] in NOMBRES}
        anns = {}
        for a in coco['annotations']:
            seg = a.get('segmentation')
            if a['category_id'] in cat_a_idx and isinstance(seg, list) and seg and len(seg[0]) >= 6:
                anns.setdefault(a['image_id'], []).append(a)
        dir_img = os.path.join(DIR_DS, split, 'images'); dir_lbl = os.path.join(DIR_DS, split, 'labels')
        shutil.rmtree(dir_img, ignore_errors=True); shutil.rmtree(dir_lbl, ignore_errors=True)
        os.makedirs(dir_img); os.makedirs(dir_lbl)
        for img in coco['images']:
            origen = os.path.join(dir_coco, split, img['file_name'])
            if not os.path.exists(origen):
                continue
            shutil.copy(origen, dir_img)
            W, H = img['width'], img['height']
            lineas = []
            for a in anns.get(img['id'], []):
                for poli in a['segmentation']:
                    pts = np.array(poli, float).reshape(-1, 2) / [W, H]
                    lineas.append(str(cat_a_idx[a['category_id']]) + ' ' + ' '.join(f'{v:.6f}' for v in np.clip(pts, 0, 1).ravel()))
            open(os.path.join(dir_lbl, os.path.splitext(img['file_name'])[0] + '.txt'), 'w').write('\n'.join(lineas))
            convertidas += 1
    return convertidas

if FUENTE_MASCARAS is None:
    DIR_COCO = '/content/dataset_m2_coco'
    shutil.rmtree(DIR_COCO, ignore_errors=True)
    try:
        version.download('coco-segmentation', location=DIR_COCO)
        tiene_segmentos = False
        for split in SPLITS:
            ruta_json = os.path.join(DIR_COCO, split, '_annotations.coco.json')
            if os.path.exists(ruta_json):
                anns = json.load(open(ruta_json, encoding='utf-8'))['annotations']
                tiene_segmentos |= any(isinstance(a.get('segmentation'), list) and a['segmentation'] and len(a['segmentation'][0]) >= 6 for a in anns)
        if tiene_segmentos:
            n = coco_a_yolo_seg(DIR_COCO)
            poligonos, cajas = contar_lineas(DIR_DS)
            print(f'COCO → YOLO-seg: {n} imágenes convertidas ({poligonos} polígonos)')
            if poligonos > 0:
                FUENTE_MASCARAS = 'COCO'
        else:
            print('La exportación COCO no trae polígonos.')
    except Exception as e:
        print('No se pudo usar la exportación COCO:', e)

# 3) Último recurso: máscaras con SAM a partir de las cajas.
if FUENTE_MASCARAS is None:
    from ultralytics import SAM
    sam = SAM('sam2.1_b.pt')
    convertidas = 0
    for split in SPLITS:
        imgs = sorted(glob.glob(os.path.join(DIR_DS, split, 'images', '*')))
        for k, ruta in enumerate(imgs):
            lbl = os.path.join(DIR_DS, split, 'labels', os.path.splitext(os.path.basename(ruta))[0] + '.txt')
            if not os.path.exists(lbl):
                continue
            filas = [l.split() for l in open(lbl) if len(l.split()) == 5]
            if not filas:
                continue
            im = cv2.imread(ruta); H, W = im.shape[:2]
            cajas_px = [[(float(x) - float(w) / 2) * W, (float(y) - float(h) / 2) * H,
                         (float(x) + float(w) / 2) * W, (float(y) + float(h) / 2) * H] for _, x, y, w, h in filas]
            r = sam(im, bboxes=cajas_px, verbose=False, device=DEVICE)[0]
            lineas = []
            for fila, mask in zip(filas, r.masks.data.cpu().numpy() if r.masks is not None else []):
                cont, _ = cv2.findContours((mask > 0.5).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if not cont:
                    continue
                c = max(cont, key=cv2.contourArea)
                c = cv2.approxPolyDP(c, 0.002 * cv2.arcLength(c, True), True).reshape(-1, 2)
                if len(c) < 3:
                    continue
                pts = c / [mask.shape[1], mask.shape[0]]
                lineas.append(fila[0] + ' ' + ' '.join(f'{v:.6f}' for v in np.clip(pts, 0, 1).ravel()))
            # Siempre se reescribe: un archivo con cajas y polígonos mezclados rompe el entrenamiento seg.
            open(lbl, 'w').write('\n'.join(lineas))
            convertidas += bool(lineas)
            if k % 500 == 0:
                print(f'  SAM {split}: {k}/{len(imgs)}')
    print(f'SAM: {convertidas} imágenes convertidas a polígonos')
    FUENTE_MASCARAS = 'SAM'

ES_SEG = True
print(f'Modo: segmentación ({FUENTE_MASCARAS})')
print('Dataset en', DIR_DS)

## 4. Revisar clases y normalizarlas
El servidor de inferencia traduce los nombres de clase a las 5 categorías de la tarifa (`moto, auto, minibus, camion, bus`).
Esta celda renombra las clases del dataset a esos nombres canónicos **sin cambiar los índices**, para que el modelo entrenado sea compatible con el servidor.
Si aparece una clase desconocida, agrégala en `SINONIMOS` y vuelve a ejecutar.

In [ ]:
CATEGORIAS = ['moto', 'auto', 'minibus', 'camion', 'bus']
SINONIMOS = {
    'moto': 'moto', 'motorcycle': 'moto', 'motorbike': 'moto', 'motocicleta': 'moto', 'scooter': 'moto',
    'auto': 'auto', 'car': 'auto', 'sedan': 'auto', 'automovil': 'auto', 'automóvil': 'auto',
    'auto particular': 'auto', 'vagoneta': 'auto', 'suv': 'auto', 'pickup': 'auto', 'jeep': 'auto', 'taxi': 'auto',
    'minibus': 'minibus', 'minibús': 'minibus', 'van': 'minibus', 'microbus': 'minibus', 'micro': 'minibus',
    'camion': 'camion', 'camión': 'camion', 'truck': 'camion', 'trailer': 'camion', 'volqueta': 'camion',
    'bus': 'bus', 'autobus': 'bus', 'autobús': 'bus', 'flota': 'bus',
    # Dataset 'car-motorcycle-bus-truck' de Roboflow Universe (clases con prefijo)
    'n-lcv': 'minibus', 'lcv': 'minibus', 'o-bus': 'bus', 'p-truck': 'camion', 'p-ambulance': 'minibus',
    'tractor123': 'camion', 'tractor': 'camion', 'rickshaw123': 'moto', 'rickshaw': 'moto',
}

ruta_yaml = os.path.join(DIR_DS, 'data.yaml')
cfg = yaml.safe_load(open(ruta_yaml, encoding='utf-8'))
nombres = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]

canonicos, desconocidas = [], []
for n in nombres:
    c = SINONIMOS.get(str(n).strip().lower())
    if c is None:
        desconocidas.append(n)
    canonicos.append(c or str(n))
print('Clases del dataset :', nombres)
print('Clases normalizadas:', canonicos)
if desconocidas:
    raise ValueError(f'Clases sin categoría de tarifa: {desconocidas}. Agrégalas a SINONIMOS.')
faltan = [c for c in CATEGORIAS if c not in canonicos]
if faltan:
    print('ADVERTENCIA: el dataset no tiene ejemplos de', faltan, '- esas categorías no se podrán detectar.')

# Rutas absolutas: Roboflow a veces escribe ../train/images y Ultralytics se confunde
cfg.update({'path': DIR_DS, 'train': 'train/images', 'val': 'valid/images',
            'names': canonicos, 'nc': len(canonicos)})
if os.path.isdir(os.path.join(DIR_DS, 'test', 'images')):
    cfg['test'] = 'test/images'
yaml.safe_dump(cfg, open(ruta_yaml, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
DATA_YAML = ruta_yaml

# Conteo de imágenes e instancias por clase
conteo = []
for split in ['train', 'valid', 'test']:
    lbls = glob.glob(os.path.join(DIR_DS, split, 'labels', '*.txt'))
    inst = {c: 0 for c in canonicos}
    for f in lbls:
        for linea in open(f):
            if linea.strip():
                inst[canonicos[int(linea.split()[0])]] += 1
    conteo.append({'split': split, 'imágenes': len(lbls), **inst})
display(pd.DataFrame(conteo).set_index('split'))

## 5. Ver algunas etiquetas
Comprueba que los polígonos están bien antes de gastar GPU entrenando.

In [ ]:
import random
import numpy as np
COLORES = {'moto': (255,165,0), 'auto': (59,130,246), 'minibus': (16,185,129), 'camion': (168,85,247), 'bus': (239,68,68)}
imgs = sorted(glob.glob(os.path.join(DIR_DS, 'train', 'images', '*')))
random.seed(1)
muestra = random.sample(imgs, min(6, len(imgs)))
fig, ax = plt.subplots(2, 3, figsize=(18, 10)); ax = ax.ravel()
for a in ax: a.axis('off')
for a, ruta in zip(ax, muestra):
    im = cv2.cvtColor(cv2.imread(ruta), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    capa = im.copy()
    lbl = ruta.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(lbl):
        for linea in open(lbl):
            v = linea.split()
            if len(v) < 5: continue
            c = canonicos[int(v[0])]
            if len(v) == 5:  # etiqueta de detección: clase xc yc w h
                xc, yc, w, h = (float(x) for x in v[1:])
                p1 = (int((xc - w / 2) * W), int((yc - h / 2) * H)); p2 = (int((xc + w / 2) * W), int((yc + h / 2) * H))
                cv2.rectangle(capa, p1, p2, COLORES.get(c, (200,200,200)), -1)
                cv2.putText(im, c, p1, cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
                continue
            pts = (np.array(v[1:], float).reshape(-1, 2) * [W, H]).astype(np.int32)
            cv2.fillPoly(capa, [pts], COLORES.get(c, (200,200,200)))
            cv2.putText(im, c, tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    a.imshow(cv2.addWeighted(capa, 0.45, im, 0.55, 0)); a.set_title(os.path.basename(ruta)[:40])
plt.tight_layout(); plt.show()

## 6. Entrenar YOLO11-seg
`yolo11s-seg` equilibra precisión y velocidad en una T4. Con pocas imágenes (<300) 60 a 80 épocas suelen bastar; `patience` detiene el entrenamiento si deja de mejorar.

In [ ]:
MODELO_BASE = 'yolo11n-seg.pt'   # siempre segmentación de instancias (la rúbrica pide máscaras en M2)
N_TRAIN = len(glob.glob(os.path.join(DIR_DS, 'train', 'images', '*')))
EPOCAS = 30                       # con parada temprana (patience=10)
IMGSZ = 640
BATCH = 16                       # bájalo a 8 si hay error de memoria

# Los pesos se guardan en Drive en cada época: si Colab se desconecta, vuelve a ejecutar esta celda y continúa.
RUNS = os.path.join(DIR_DRIVE, 'runs_m2')
ULTIMO = os.path.join(RUNS, 'm2', 'weights', 'last.pt')
print(f'Modo: segmentación ({FUENTE_MASCARAS}) | {N_TRAIN} imágenes de entrenamiento -> {EPOCAS} épocas con {MODELO_BASE}')

t0 = time.time()
if os.path.exists(ULTIMO):
    print('Reanudando desde', ULTIMO)
    YOLO(ULTIMO).train(resume=True)
else:
    modelo = YOLO(MODELO_BASE)
    modelo.train(data=DATA_YAML, epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH, patience=10,
                 device=DEVICE, project=RUNS, name='m2', exist_ok=True, plots=True)
RUTA_BEST = os.path.join(RUNS, 'm2', 'weights', 'best.pt')
print(f'Entrenamiento terminado en {(time.time() - t0) / 60:.1f} min ->', RUTA_BEST)


## 7. Métricas (para el informe)
Se evalúa el mejor peso sobre validación: **mAP50 y mAP50-95 de cajas y de máscaras**, por clase, más la matriz de confusión y las curvas de entrenamiento.

In [ ]:
mejor = YOLO(RUTA_BEST)
val = mejor.val(data=DATA_YAML, split='val', imgsz=IMGSZ, device=DEVICE, plots=True,
                project=RUNS, name='m2_val', exist_ok=True)

resumen = {
    'dataset': f'{RF_WORKSPACE}/{RF_PROJECT}/v{RF_VERSION}',
    'tarea': 'segmentacion' if ES_SEG else 'deteccion',
    'modelo_base': MODELO_BASE, 'epocas': EPOCAS, 'imgsz': IMGSZ, 'imagenes_train': N_TRAIN,
    'box_mAP50': round(float(val.box.map50), 4), 'box_mAP50_95': round(float(val.box.map), 4),
    'precision_box': round(float(val.box.mp), 4), 'recall_box': round(float(val.box.mr), 4),
    'velocidad_ms': {k: round(v, 2) for k, v in val.speed.items()},
}
if ES_SEG:
    resumen.update({'mask_mAP50': round(float(val.seg.map50), 4), 'mask_mAP50_95': round(float(val.seg.map), 4),
                    'precision_mask': round(float(val.seg.mp), 4), 'recall_mask': round(float(val.seg.mr), 4)})
print(json.dumps(resumen, indent=2, ensure_ascii=False))

por_clase = []
for i, c in enumerate(val.box.ap_class_index):
    fila = {'clase': mejor.names[int(c)], 'box_mAP50': round(float(val.box.ap50[i]), 3),
            'box_mAP50_95': round(float(val.box.ap[i]), 3)}
    if ES_SEG:
        fila.update({'mask_mAP50': round(float(val.seg.ap50[i]), 3), 'mask_mAP50_95': round(float(val.seg.ap[i]), 3)})
    por_clase.append(fila)
tabla_clases = pd.DataFrame(por_clase).set_index('clase')
display(tabla_clases)

for img in [os.path.join(RUNS, 'm2', 'results.png'), os.path.join(RUNS, 'm2_val', 'confusion_matrix_normalized.png')]:
    if os.path.exists(img):
        plt.figure(figsize=(14, 8)); plt.imshow(plt.imread(img)); plt.axis('off'); plt.show()


## 8. Prueba con una imagen propia
Sube una foto que **no** esté en el dataset. Se muestra la máscara y la categoría que usará la tarifa, con la misma lógica que el servidor de inferencia (el vehículo de mayor confianza).

In [ ]:
subidas = files.upload()
for nombre in subidas:
    r = mejor.predict(nombre, conf=0.25, device=DEVICE, verbose=False)[0]
    if len(r.boxes):
        i = int(r.boxes.conf.argmax())
        cat, conf = r.names[int(r.boxes.cls[i])], float(r.boxes.conf[i])
        area = float(r.masks.data[i].sum() / r.masks.data[i].numel()) if r.masks is not None else None
        texto_area = f'{area:.3f}' if area is not None else 'n/d (detección)'
        print(f'{nombre}: categoria={cat} confianza={conf:.2f} area_mascara={texto_area}')
    else:
        print(f'{nombre}: sin vehículos detectados')
    plt.figure(figsize=(10, 7)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off'); plt.show()

## 9. Guardar en Google Drive
Copia `m2_best.pt` y las métricas a `MyDrive/peaje_modelos/`. El notebook `servidor_inferencia_peaje_colab.ipynb` lo toma desde ahí automáticamente.
Las métricas y gráficas también sirven para la carpeta de evidencias del informe.

In [ ]:
shutil.copy(RUTA_BEST, os.path.join(DIR_DRIVE, 'm2_best.pt'))
dir_ev = os.path.join(DIR_DRIVE, 'evidencias_m2')
os.makedirs(dir_ev, exist_ok=True)
json.dump(resumen, open(os.path.join(dir_ev, 'metricas_m2.json'), 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
tabla_clases.to_csv(os.path.join(dir_ev, 'metricas_m2_por_clase.csv'))
for patron in [os.path.join(RUNS, 'm2', '*.png'), os.path.join(RUNS, 'm2', 'results.csv'), os.path.join(RUNS, 'm2_val', '*.png')]:
    for f in glob.glob(patron):
        shutil.copy(f, os.path.join(dir_ev, os.path.basename(f)))
print('Guardado:')
for f in sorted(os.listdir(DIR_DRIVE)):
    print(' ', f)